In [ ]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path

import rasterio
from rasterio.plot import show
from rasterio.mask import mask
from rasterio.merge import merge
import rasterstats
from rasterstats import zonal_stats

In [ ]:
data_root = '../data/elevation'
elevation_ds = rasterio.open(os.path.join(data_root, "merged.tif"))


In [ ]:
def apply_elevation_avg(row):
    geom = row['geometry']
    # Create mask from polygon
    masked_data, masked_transform = mask(elevation_ds, [geom], crop=True, nodata=np.nan)
    return np.nanmean(masked_data)

In [ ]:
results_root = '../outputs'
n_clusters = 2

dis_folders = [
    '4775',
    '4859',
]
for dis_no in dis_folders:
    dis_gpd = gpd.read_file(os.path.join(results_root, f'{dis_no}/cluster_{n_clusters}', f'clustering_result_w_grdi_{dis_no}_correct_label.geojson'))
    
    # Calculate average elevation for each polygon
    dis_gpd['avg_elevation'] = dis_gpd.apply(apply_elevation_avg, axis=1)
    
    dis_gpd.to_file(
        os.path.join(results_root, f'{dis_no}/cluster_{n_clusters}', f"clustering_result_w_grdi_{dis_no}_correct_label_elevation.geojson"),
        driver='GeoJSON'
    )


In [ ]:
dis_gpd